# Manuka 04 Preprocess Experiment

Daily conversation JSON is converted to speaker-tagged plain text, then combined with `pretrain.zip` for plain language-model training. Messenger CSV is added to tokenizer training and exported separately for SFT.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Run once per fresh Colab runtime if sentencepiece is missing.
try:
    import sentencepiece  # noqa: F401
except ImportError:
    %pip -q install sentencepiece


In [3]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp')
PREP_DIR = DATA_ROOT / 'prepared'
MODEL_DIR = DATA_ROOT / 'manuka-model'

DATASET_ROOT = Path('/content/drive/MyDrive/text_dataset/dataset')
DAILY_ZIP_NAMES = [
    '일상대화_데이터.zip',
    '일상대화_데이터.zip',
]
PRETRAIN_ZIP_NAME = '뉴스_데이터.zip'
MESSENGER_ZIP_NAMES = [
    '메신저.zip',
    '메신저_데이터.zip',
    'messenger.zip',
]


def candidate_paths(file_names):
    folders = [
        DATASET_ROOT,
        DATA_ROOT / 'dataset',
        DATA_ROOT,
        Path('/content/drive/MyDrive/text_dataset/dataset'),
    ]
    seen = set()
    out = []
    for folder in folders:
        for name in file_names:
            path = folder / name
            key = str(path)
            if key not in seen:
                out.append(path)
                seen.add(key)
    return out


def resolve_required_zip(label, candidates):
    for path in candidates:
        if path.exists():
            return path
    searched = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(f'Could not find {label}. Checked:\n{searched}')


DAILY_ZIP_PATH = resolve_required_zip('daily conversation zip', candidate_paths(DAILY_ZIP_NAMES))
PRETRAIN_ZIP_PATH = resolve_required_zip('pretrain.zip', candidate_paths([PRETRAIN_ZIP_NAME]))
MESSENGER_ZIP_PATH = resolve_required_zip('messenger zip', candidate_paths(MESSENGER_ZIP_NAMES))

PREP_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT:', DATA_ROOT)
print('DAILY_ZIP_PATH:', DAILY_ZIP_PATH)
print('PRETRAIN_ZIP_PATH:', PRETRAIN_ZIP_PATH)
print('MESSENGER_ZIP_PATH:', MESSENGER_ZIP_PATH)
print('PREP_DIR:', PREP_DIR)
print('MODEL_DIR:', MODEL_DIR)


DATA_ROOT: /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp
DAILY_ZIP_PATH: /content/drive/MyDrive/text_dataset/dataset/일상대화_데이터.zip
PRETRAIN_ZIP_PATH: /content/drive/MyDrive/text_dataset/dataset/뉴스_데이터.zip
MESSENGER_ZIP_PATH: /content/drive/MyDrive/text_dataset/dataset/메신저_데이터.zip
PREP_DIR: /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp/prepared
MODEL_DIR: /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp/manuka-model


# Data Load


In [4]:
# === Speaker-tagged daily dialog + pretrain.zip + messenger tokenizer corpus ===
import ast
import csv
import gzip
import html
import io
import json
import re
import zipfile
from collections import Counter

CONTEXT_LENGTH = 512
MAX_LEN = CONTEXT_LENGTH
MAX_CORPUS_LINE_CHARS = 8000
MIN_TEXT_CHARS = 2
MIN_HANGUL_RATIO = 0.02
MAX_SPEAKER_TAGS = 64

TEXT_FIELD_PRIORITY = ('form', 'original_form', 'text', 'content', 'sentence', 'body')
SOURCE_PREFIX = {
    'daily_dialog': '<daily>',
    'pretrain': '<pretrain>',
    'messenger_sft': '<messenger>',
}
USER_DEFINED_SYMBOLS = (
    ['<daily>', '<pretrain>', '<messenger>', '<turn>', '<spk_other>']
    + [f'<spk_{i:02d}>' for i in range(MAX_SPEAKER_TAGS)]
)
PROTECTED_SYMBOLS = ['<sep>'] + USER_DEFINED_SYMBOLS

SPACE_RGX = re.compile(r'\s+')
TAG_RGX = re.compile(r'<[^>]+>')
KOR_RGX = re.compile(r'[가-힣ㄱ-ㅎㅏ-ㅣ]')
DECODINGS = ('utf-8-sig', 'utf-8', 'cp949', 'euc-kr')

CORPUS_PATH = Path('plain_corpus_04_preprocess_exp.txt')
CORPUS_META_PATH = Path('plain_corpus_meta_04_preprocess_exp.jsonl.gz')
SFT_PAIRS_PATH = Path('messenger_sft_pairs.jsonl.gz')
SFT_TOKENIZED_PATH = Path('messenger_sft_tokenized_examples.jsonl.gz')
PLAIN_LM_KINDS = {'daily_dialog', 'pretrain'}
SFT_PAIR_MODE = 'single_turn_previous_to_next'

source_doc_counts = Counter()
source_line_counts = Counter()
source_char_counts = Counter()
source_speaker_counts = Counter()
bad_files = []


def clean_text(s: str) -> str:
    s = html.unescape(str(s or ''))
    placeholders = {}
    for i, symbol in enumerate(PROTECTED_SYMBOLS):
        placeholder = f'@@SPM_SYMBOL_{i}@@'
        if symbol in s:
            s = s.replace(symbol, f' {placeholder} ')
            placeholders[placeholder] = symbol
    s = TAG_RGX.sub(' ', s)
    for placeholder, symbol in placeholders.items():
        s = s.replace(placeholder, symbol)
    s = s.replace('\u200b', ' ').replace('\ufeff', ' ').strip()
    return SPACE_RGX.sub(' ', s)


def is_usable_text(s: str) -> bool:
    s = clean_text(s)
    if len(s) < MIN_TEXT_CHARS:
        return False
    hangul = len(KOR_RGX.findall(s))
    return hangul >= max(1, int(len(s) * MIN_HANGUL_RATIO))


def pick_text(obj) -> str:
    if isinstance(obj, str):
        return clean_text(obj)
    if not isinstance(obj, dict):
        return ''
    for field in TEXT_FIELD_PRIORITY:
        text = clean_text(obj.get(field, ''))
        if text:
            return text
    return ''


def decode_bytes(raw: bytes):
    for encoding in DECODINGS:
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            pass
    return raw.decode('utf-8', errors='replace'), 'utf-8-replace'


def load_json_member(zf, name):
    text, encoding = decode_bytes(zf.read(name))
    return json.loads(text), encoding


def iter_documents(data):
    if isinstance(data, dict):
        docs = data.get('document')
        if isinstance(docs, list):
            for doc in docs:
                yield doc
            return
        if isinstance(docs, dict):
            yield docs
            return
        yield data
    elif isinstance(data, list):
        for item in data:
            yield item


def iter_text_nodes(node):
    if isinstance(node, str):
        text = clean_text(node)
        if text:
            yield text
        return
    if isinstance(node, list):
        for item in node:
            yield from iter_text_nodes(item)
        return
    if not isinstance(node, dict):
        return

    direct = pick_text(node)
    if direct:
        yield direct
        return

    for key in ('paragraph', 'paragraphs', 'sentence', 'sentences', 'text', 'content', 'body'):
        if key in node:
            yield from iter_text_nodes(node[key])


def utterance_sort_key(utt):
    start = utt.get('start') if isinstance(utt, dict) else None
    if isinstance(start, str):
        try:
            start = float(start)
        except ValueError:
            start = float('inf')
    if not isinstance(start, (int, float)):
        start = float('inf')
    return (start, str(utt.get('id', '')) if isinstance(utt, dict) else '')


def speaker_tag(index: int) -> str:
    if index < MAX_SPEAKER_TAGS:
        return f'<spk_{index:02d}>'
    return '<spk_other>'


def iter_daily_records(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if not name.lower().endswith('.json'):
                continue
            try:
                data, _ = load_json_member(zf, name)
                for doc in iter_documents(data):
                    if not isinstance(doc, dict):
                        continue
                    utterances = doc.get('utterance', [])
                    if not isinstance(utterances, list):
                        continue

                    speaker_map = {}
                    pieces = []
                    for utt in sorted(utterances, key=utterance_sort_key):
                        if not isinstance(utt, dict):
                            continue
                        text = pick_text(utt)
                        if not is_usable_text(text):
                            continue
                        raw_speaker = str(utt.get('speaker_id') or utt.get('speaker') or 'unknown')
                        if raw_speaker not in speaker_map:
                            speaker_map[raw_speaker] = speaker_tag(len(speaker_map))
                        pieces.append(f'<turn> {speaker_map[raw_speaker]} {text}')

                    if pieces:
                        yield {
                            'kind': 'daily_dialog',
                            'source': name,
                            'doc_id': str(doc.get('id', '')),
                            'speaker_count': len(speaker_map),
                            'utterance_count': len(pieces),
                            'text': ' '.join(pieces),
                        }
            except Exception as exc:
                bad_files.append({'kind': 'daily_dialog', 'source': name, 'error': repr(exc)})


def record_from_text(kind, source, doc_id, text):
    text = clean_text(text)
    if not is_usable_text(text):
        return None
    return {'kind': kind, 'source': source, 'doc_id': str(doc_id or ''), 'text': text}


def iter_pretrain_records(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            lower = name.lower()
            if lower.endswith('/'):
                continue
            try:
                if lower.endswith('.json'):
                    data, _ = load_json_member(zf, name)
                    for doc in iter_documents(data):
                        texts = [text for text in iter_text_nodes(doc) if is_usable_text(text)]
                        if not texts:
                            continue
                        doc_id = doc.get('id', '') if isinstance(doc, dict) else ''
                        yield record_from_text('pretrain', name, doc_id, ' '.join(texts))
                elif lower.endswith('.jsonl'):
                    text, _ = decode_bytes(zf.read(name))
                    for line_no, line in enumerate(text.splitlines(), 1):
                        line = line.strip()
                        if not line:
                            continue
                        obj = json.loads(line)
                        texts = [text for text in iter_text_nodes(obj) if is_usable_text(text)]
                        if texts:
                            yield record_from_text('pretrain', name, f'{name}:{line_no}', ' '.join(texts))
                elif lower.endswith(('.txt', '.text', '.md')):
                    text, _ = decode_bytes(zf.read(name))
                    yield record_from_text('pretrain', name, name, text)
            except Exception as exc:
                bad_files.append({'kind': 'pretrain', 'source': name, 'error': repr(exc)})


def parse_messenger_speaker_id(raw_speaker: str) -> str:
    try:
        speaker = ast.literal_eval(raw_speaker or '{}')
        if isinstance(speaker, dict):
            return str(speaker.get('id') or 'unknown')
    except (SyntaxError, ValueError):
        pass
    return str(raw_speaker or 'unknown')


def iter_messenger_dialogs(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if not name.lower().endswith('.csv'):
                continue
            try:
                text, _ = decode_bytes(zf.read(name))
                reader = csv.DictReader(io.StringIO(text))
                current_doc_id = None
                current_turns = []
                speaker_map = {}

                def flush():
                    if current_doc_id and current_turns:
                        yield {
                            'kind': 'messenger_sft',
                            'source': name,
                            'doc_id': current_doc_id,
                            'speaker_count': len(speaker_map),
                            'utterance_count': len(current_turns),
                            'turns': current_turns[:],
                            'text': ' '.join(turn['tagged'] for turn in current_turns),
                        }

                for row in reader:
                    doc_id = str(row.get('doc_id') or row.get('file_id') or '').strip()
                    if not doc_id:
                        continue
                    if current_doc_id is not None and doc_id != current_doc_id:
                        yield from flush()
                        current_turns = []
                        speaker_map = {}
                    current_doc_id = doc_id

                    text = clean_text(row.get('form') or row.get('original_form') or '')
                    if not is_usable_text(text):
                        continue
                    raw_speaker = parse_messenger_speaker_id(row.get('speaker', ''))
                    if raw_speaker not in speaker_map:
                        speaker_map[raw_speaker] = speaker_tag(len(speaker_map))
                    tag = speaker_map[raw_speaker]
                    current_turns.append({'speaker': tag, 'text': text, 'tagged': f'<turn> {tag} {text}'})

                yield from flush()
            except Exception as exc:
                bad_files.append({'kind': 'messenger_sft', 'source': name, 'error': repr(exc)})


def iter_sft_pairs(dialogs):
    for dialog in dialogs:
        turns = dialog.get('turns', [])
        if len(turns) < 2:
            continue
        for i in range(1, len(turns)):
            prompt = turns[i - 1]['text']
            response = turns[i]['text']
            if is_usable_text(prompt) and is_usable_text(response):
                yield {
                    'prompt': prompt,
                    'response': response,
                    'prompt_speaker': turns[i - 1].get('speaker', ''),
                    'response_speaker': turns[i].get('speaker', ''),
                    'kind': 'messenger_sft',
                    'doc_id': dialog.get('doc_id', ''),
                    'source': dialog.get('source', ''),
                }


def split_for_corpus(text: str, max_chars: int = MAX_CORPUS_LINE_CHARS):
    words = clean_text(text).split()
    current = []
    current_len = 0
    for word in words:
        add_len = len(word) + (1 if current else 0)
        if current and current_len + add_len > max_chars:
            yield ' '.join(current)
            current = [word]
            current_len = len(word)
        else:
            current.append(word)
            current_len += add_len
    if current:
        yield ' '.join(current)


def write_record(record, corpus_f, meta_f):
    if record is None:
        return 0
    kind = record['kind']
    prefix = SOURCE_PREFIX.get(kind, '<pretrain>')
    written = 0
    for chunk in split_for_corpus(record['text']):
        if not is_usable_text(chunk):
            continue
        line = f'{prefix} {chunk}'
        corpus_f.write(line + '\n')
        meta_f.write(json.dumps({
            'kind': kind,
            'source': record.get('source', ''),
            'doc_id': record.get('doc_id', ''),
        }, ensure_ascii=False) + '\n')
        source_line_counts[kind] += 1
        source_char_counts[kind] += len(line)
        written += 1
    if written:
        source_doc_counts[kind] += 1
        if kind == 'daily_dialog':
            source_speaker_counts[str(record.get('speaker_count', 0))] += 1
    return written


sft_pair_count = 0
with open(CORPUS_PATH, 'w', encoding='utf-8') as corpus_f, gzip.open(CORPUS_META_PATH, 'wt', encoding='utf-8') as meta_f, gzip.open(SFT_PAIRS_PATH, 'wt', encoding='utf-8') as sft_f:
    for record in iter_daily_records(DAILY_ZIP_PATH):
        write_record(record, corpus_f, meta_f)
    for record in iter_pretrain_records(PRETRAIN_ZIP_PATH):
        write_record(record, corpus_f, meta_f)
    for record in iter_messenger_dialogs(MESSENGER_ZIP_PATH):
        write_record(record, corpus_f, meta_f)
        for pair in iter_sft_pairs([record]):
            sft_f.write(json.dumps(pair, ensure_ascii=False) + '\n')
            sft_pair_count += 1

corpus_line_count = sum(source_line_counts.values())
if corpus_line_count == 0:
    raise ValueError('No usable corpus lines were produced. Check the dataset encodings and JSON schemas.')

print('Corpus path:', CORPUS_PATH)
print('Corpus meta path:', CORPUS_META_PATH)
print('Source documents:', dict(source_doc_counts))
print('Corpus lines:', dict(source_line_counts))
print('Corpus chars:', dict(source_char_counts))
print('Daily speaker-count docs:', dict(source_speaker_counts))
print('Messenger SFT pairs:', sft_pair_count)
print('Bad files:', len(bad_files))


Corpus path: plain_corpus_04_preprocess_exp.txt
Corpus meta path: plain_corpus_meta_04_preprocess_exp.jsonl.gz
Source documents: {'daily_dialog': 14229, 'pretrain': 1206618, 'messenger_sft': 3836}
Corpus lines: {'daily_dialog': 27711, 'pretrain': 1206636, 'messenger_sft': 5152}
Corpus chars: {'daily_dialog': 152190656, 'pretrain': 1214625001, 'messenger_sft': 18659856}
Daily speaker-count docs: {'1': 380, '2': 11520, '3': 1554, '4': 775}
Messenger SFT pairs: 660455
Bad files: 0


# Tokenizer


In [5]:
# SentencePiece tokenizer trained on the same merged plain-text corpus used by the model.
import os
import sentencepiece as spm

VOCAB_TARGET = 32000
TOKENIZER_PREFIX = 'spm32k'

spm.SentencePieceTrainer.train(
    input=str(CORPUS_PATH),
    model_prefix=TOKENIZER_PREFIX,
    vocab_size=VOCAB_TARGET,
    model_type='unigram',
    character_coverage=0.9998,
    pad_id=0, bos_id=1, eos_id=3, unk_id=4,
    control_symbols=['<sep>'],
    user_defined_symbols=USER_DEFINED_SYMBOLS,
    byte_fallback=True,
    hard_vocab_limit=False,
    num_threads=max(1, os.cpu_count() or 1),
)

sp = spm.SentencePieceProcessor(model_file=f'{TOKENIZER_PREFIX}.model')

PAD = sp.pad_id(); BOS = sp.bos_id(); EOS = sp.eos_id(); UNK = sp.unk_id()
SEP = sp.piece_to_id('<sep>')
VOCAB_SIZE = sp.get_piece_size()
SPECIALS = {'PAD': PAD, 'BOS': BOS, 'SEP': SEP, 'EOS': EOS, 'UNK': UNK}


def encode_text(s: str):
    return sp.encode(clean_text(s), out_type=int)


def decode_text(ids):
    drop = {PAD, BOS, SEP, EOS, UNK}
    return sp.decode([int(i) for i in ids if int(i) not in drop])


print('Tokenizer vocab size:', VOCAB_SIZE, SPECIALS)


Tokenizer vocab size: 32000 {'PAD': 0, 'BOS': 1, 'SEP': 2, 'EOS': 3, 'UNK': 4}


# Tokenized Examples


In [6]:
# Plain LM rows exclude messenger_sft; messenger is tokenized separately for SFT.
from itertools import zip_longest

TOKENIZED_PATH = Path('tokenized_examples.jsonl.gz')
MIN_CHUNK_TOKENS = 16
CHUNK_OVERLAP = 0


def iter_lm_chunks(ids):
    if len(ids) <= MAX_LEN:
        if len(ids) >= MIN_CHUNK_TOKENS:
            yield ids
        return

    body = ids[1:-1]
    max_body = MAX_LEN - 2
    stride = max(1, max_body - CHUNK_OVERLAP)
    for start in range(0, len(body), stride):
        chunk_body = body[start:start + max_body]
        if len(chunk_body) + 2 >= MIN_CHUNK_TOKENS:
            yield [BOS] + chunk_body + [EOS]
        if start + max_body >= len(body):
            break


tokenized_kind_counts = Counter()
tokenized_examples_count = 0
tokens_per_epoch = 0
max_token_length = 0

with open(CORPUS_PATH, 'r', encoding='utf-8') as corpus_f, gzip.open(CORPUS_META_PATH, 'rt', encoding='utf-8') as meta_f, gzip.open(TOKENIZED_PATH, 'wt', encoding='utf-8') as out_f:
    for line_no, (line, meta_line) in enumerate(zip_longest(corpus_f, meta_f), 1):
        if line is None or meta_line is None:
            raise ValueError(f'Corpus/meta line count mismatch around line {line_no}')
        line = clean_text(line)
        if not line:
            continue
        meta = json.loads(meta_line)
        if meta.get('kind') not in PLAIN_LM_KINDS:
            continue
        ids = [BOS] + encode_text(line) + [EOS]
        for chunk in iter_lm_chunks(ids):
            row = {
                'input_ids': chunk,
                'labels': chunk[:],
                'kind': meta.get('kind', 'unknown'),
                'doc_id': meta.get('doc_id', ''),
                'source': meta.get('source', ''),
            }
            out_f.write(json.dumps(row, ensure_ascii=False) + '\n')
            tokenized_kind_counts[row['kind']] += 1
            tokenized_examples_count += 1
            tokens_per_epoch += len(chunk)
            max_token_length = max(max_token_length, len(chunk))

if tokenized_examples_count == 0:
    raise ValueError('No tokenized examples were produced.')

print('Tokenized path:', TOKENIZED_PATH)
print('Tokenized examples:', tokenized_examples_count)
print('Tokenized mix:', dict(tokenized_kind_counts))
print('Tokens per epoch before dynamic padding:', tokens_per_epoch)
print('Max token length:', max_token_length)


def encode_sft_pair(pair):
    prompt_ids = encode_text(pair['prompt'])
    response_ids = encode_text(pair['response']) + [EOS]

    max_response_tokens = max(8, MAX_LEN - 32)
    if len(response_ids) > max_response_tokens:
        response_ids = response_ids[:max_response_tokens]
        response_ids[-1] = EOS

    prompt_room = MAX_LEN - len(response_ids) - 2
    if prompt_room <= 0:
        return None
    prompt_ids = prompt_ids[-prompt_room:]

    input_ids = [BOS] + prompt_ids + [SEP] + response_ids
    labels = [-100] * (len(prompt_ids) + 2) + response_ids
    if len(input_ids) <= MAX_LEN and any(t != -100 for t in labels):
        return {
            'input_ids': input_ids,
            'labels': labels,
            'kind': pair.get('kind', 'messenger_sft'),
            'doc_id': pair.get('doc_id', ''),
            'source': pair.get('source', ''),
            'prompt_speaker': pair.get('prompt_speaker', ''),
            'response_speaker': pair.get('response_speaker', ''),
        }
    return None


sft_tokenized_count = 0
sft_tokens_per_epoch = 0
sft_max_token_length = 0
with gzip.open(SFT_PAIRS_PATH, 'rt', encoding='utf-8') as in_f, gzip.open(SFT_TOKENIZED_PATH, 'wt', encoding='utf-8') as out_f:
    for line in in_f:
        pair = json.loads(line)
        row = encode_sft_pair(pair)
        if row is None:
            continue
        out_f.write(json.dumps(row, ensure_ascii=False) + '\n')
        sft_tokenized_count += 1
        sft_tokens_per_epoch += len(row['input_ids'])
        sft_max_token_length = max(sft_max_token_length, len(row['input_ids']))

print('SFT pairs path:', SFT_PAIRS_PATH)
print('SFT tokenized path:', SFT_TOKENIZED_PATH)
print('SFT tokenized examples:', sft_tokenized_count)
print('SFT tokens per epoch before dynamic padding:', sft_tokens_per_epoch)
print('SFT max token length:', sft_max_token_length)


Tokenized path: tokenized_examples.jsonl.gz
Tokenized examples: 1599292
Tokenized mix: {'daily_dialog': 116312, 'pretrain': 1482980}
Tokens per epoch before dynamic padding: 550729748
Max token length: 512
SFT pairs path: messenger_sft_pairs.jsonl.gz
SFT tokenized path: messenger_sft_tokenized_examples.jsonl.gz
SFT tokenized examples: 660455
SFT tokens per epoch before dynamic padding: 10949895
SFT max token length: 512


In [7]:
# Quick non-training inspection of the first prepared row.
with gzip.open(TOKENIZED_PATH, 'rt', encoding='utf-8') as f:
    first_row = json.loads(next(f))

print('First row keys:', list(first_row.keys()))
print('First row kind:', first_row.get('kind'))
print('First row token length:', len(first_row['input_ids']))
print('First row text preview:', decode_text(first_row['input_ids'][:160]))


First row keys: ['input_ids', 'labels', 'kind', 'doc_id', 'source']
First row kind: daily_dialog
First row token length: 512
First row text preview: <daily> <turn> <spk_00> 저는 개인적으로 복사가 터졌으면 좋겠어요. <turn> <spk_00> 자 여러분들 피티알 2일차 새롭게 알게 된 것들 그리고 모르면 손해 보는 것들을 정리해 봤습니다. <turn> <spk_00> 꼭 아셔야 되는 내용들이니까 같이 한번 볼게요. <turn> <spk_00> 첫 번째로 오늘 좀 화제가 됐던 게 피티알에 대기열이 좀 생겼고 <turn> <spk_00> 굉장히 좀 대기열 시간이 길었어요. <turn> <spk_00> 피티알 서버라서 이 서버 자체가 작아서 <turn> <spk_00> 그런 게 아니겠냐 이런 의견이 대부분이긴 한데 어쨌든 사람들이 몰리고 있다. 긍정적인 반응으로 볼 수 있겠죠. <turn> <spk_00> 자 그다음에 첫 번째 여러분 템퍼링에는 레시피가 존재하는데요. <turn> 


# Save Prepared Artifacts


In [8]:
from pathlib import Path
import shutil

SAVE_COMPRESSED_CORPUS = True
PREP_DIR.mkdir(parents=True, exist_ok=True)

for src_name in [f'{TOKENIZER_PREFIX}.model', f'{TOKENIZER_PREFIX}.vocab']:
    src_path = Path(src_name)
    if not src_path.exists():
        raise FileNotFoundError(f'{src_path} not found. Run the tokenizer cell first.')
    shutil.copy2(src_path, PREP_DIR / src_path.name)
    print(f'Copied: {src_path} -> {PREP_DIR / src_path.name}')

tokenized_path = PREP_DIR / TOKENIZED_PATH.name
shutil.copy2(TOKENIZED_PATH, tokenized_path)
print(f'Copied: {TOKENIZED_PATH} -> {tokenized_path}')

corpus_meta_path = PREP_DIR / CORPUS_META_PATH.name
shutil.copy2(CORPUS_META_PATH, corpus_meta_path)
print(f'Copied: {CORPUS_META_PATH} -> {corpus_meta_path}')

sft_pairs_path = PREP_DIR / SFT_PAIRS_PATH.name
shutil.copy2(SFT_PAIRS_PATH, sft_pairs_path)
print(f'Copied: {SFT_PAIRS_PATH} -> {sft_pairs_path}')

sft_tokenized_path = PREP_DIR / SFT_TOKENIZED_PATH.name
shutil.copy2(SFT_TOKENIZED_PATH, sft_tokenized_path)
print(f'Copied: {SFT_TOKENIZED_PATH} -> {sft_tokenized_path}')

plain_corpus_file = CORPUS_PATH.name
if SAVE_COMPRESSED_CORPUS:
    plain_corpus_gz_path = PREP_DIR / 'plain_corpus.txt.gz'
    with open(CORPUS_PATH, 'rt', encoding='utf-8') as src_f, gzip.open(plain_corpus_gz_path, 'wt', encoding='utf-8') as dst_f:
        shutil.copyfileobj(src_f, dst_f)
    plain_corpus_file = plain_corpus_gz_path.name
    print(f'Compressed corpus: {CORPUS_PATH} -> {plain_corpus_gz_path}')

manifest = {
    'experiment': '04-preprocess-exp',
    'task': 'plain_language_modeling',
    'data_root': str(DATA_ROOT),
    'daily_zip_path': str(DAILY_ZIP_PATH),
    'pretrain_zip_path': str(PRETRAIN_ZIP_PATH),
    'messenger_zip_path': str(MESSENGER_ZIP_PATH),
    'prepared_dir': str(PREP_DIR),
    'model_dir': str(MODEL_DIR),
    'source_documents': dict(source_doc_counts),
    'corpus_lines': dict(source_line_counts),
    'corpus_chars': dict(source_char_counts),
    'daily_speaker_count_docs': dict(source_speaker_counts),
    'training_examples': tokenized_examples_count,
    'tokenized_examples': tokenized_examples_count,
    'example_mix': dict(tokenized_kind_counts),
    'tokens_per_epoch_before_padding': tokens_per_epoch,
    'max_token_length': max_token_length,
    'sft_pair_mode': SFT_PAIR_MODE,
    'sft_pairs': sft_pair_count,
    'sft_tokenized_examples': sft_tokenized_count,
    'sft_tokens_per_epoch_before_padding': sft_tokens_per_epoch,
    'sft_max_token_length': sft_max_token_length,
    'context_length': MAX_LEN,
    'format': 'plain LM: BOS + text chunk + EOS; labels=input_ids',
    'sft_format': 'single-turn: BOS + previous utterance text + SEP + next utterance text + EOS; labels mask prompt',
    'speaker_format': '<turn> <spk_XX> utterance text for daily_dialog records',
    'special_tokens': SPECIALS,
    'user_defined_symbols': USER_DEFINED_SYMBOLS,
    'tokenizer_model': f'{TOKENIZER_PREFIX}.model',
    'tokenized_file': tokenized_path.name,
    'sft_pairs_file': sft_pairs_path.name,
    'sft_tokenized_file': sft_tokenized_path.name,
    'plain_corpus_file': plain_corpus_file,
    'corpus_meta_file': corpus_meta_path.name,
    'bad_file_count': len(bad_files),
    'bad_files_sample': bad_files[:50],
}
with open(PREP_DIR / 'data_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print('Saved prepared artifacts to:', PREP_DIR)
print('Tokenized rows:', tokenized_examples_count)


Copied: spm32k.model -> /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp/prepared/spm32k.model
Copied: spm32k.vocab -> /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp/prepared/spm32k.vocab
Copied: tokenized_examples.jsonl.gz -> /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp/prepared/tokenized_examples.jsonl.gz
Copied: plain_corpus_meta_04_preprocess_exp.jsonl.gz -> /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp/prepared/plain_corpus_meta_04_preprocess_exp.jsonl.gz
Copied: messenger_sft_pairs.jsonl.gz -> /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp/prepared/messenger_sft_pairs.jsonl.gz
Copied: messenger_sft_tokenized_examples.jsonl.gz -> /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp/prepared/messenger_sft_tokenized_examples.jsonl.gz
Compressed corpus: plain_corpus_04_preprocess_exp.txt -> /content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp/prepared/plain_corpus.txt.gz
Saved prepared artifacts to